# Football-LLM · Notebook 3 · ICL + CoT Baselines

**ECE 590.09 Final Project · Zanwen Fu**

Third notebook in the pipeline, following:
1. `train_colab.ipynb` — QLoRA fine-tuning (done, adapter at `zanwenfu/football-llm-qlora`)
2. `eval_harness.ipynb` — base vs. FT vs. baseline comparison (done, FT = 52.3% / 29.7% / 1.29)

This notebook runs the two gradient-free baselines from the proposal:
- **In-Context Learning (ICL)** — base Llama 3.1 8B + 5 few-shot demos, stratified 2 home / 2 away / 1 draw
- **Chain-of-Thought (CoT)** — base Llama 3.1 8B + 5-step reasoning template

Both are evaluated on the full 128-sample 2022 eval set and broken down by **named vs. anonymized** to test the central claim: fine-tuning's gain is primarily a de-biasing effect (suppressing pre-training priors on team names) rather than learned statistical reasoning. If that's right, ICL and CoT — which cannot update parameters — should fail to achieve the same named/anon inversion that FT shows.

**Setup:** identical to `eval_harness.ipynb` — data cloned from GitHub repo, same parser, same generation config.

**Outputs:** `icl_results.json`, `cot_results.json` saved locally. Metrics printed at the end for direct comparison with the existing FT/base/baseline table.


## 1. Setup

In [1]:
%pip install -q transformers accelerate bitsandbytes peft datasets

from huggingface_hub import notebook_login
notebook_login()

In [2]:
import torch
import json
import re
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
print(f"torch: {torch.__version__}, cuda: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

torch: 2.10.0+cu128, cuda: True
GPU: Tesla T4


## 2. Load eval + train data from the repo

In [3]:
!git clone https://github.com/zanwenfu/football-llm.git 2>/dev/null || true

eval_dataset = load_dataset(
    "json", data_files="football-llm/data/training/eval.jsonl", split="train"
)
train_dataset = load_dataset(
    "json", data_files="football-llm/data/training/train.jsonl", split="train"
)
print(f"Eval samples:  {len(eval_dataset)}")
print(f"Train samples: {len(train_dataset)}")

# Split eval into named vs anonymized (same convention as eval_harness.ipynb)
named_eval, anon_eval = [], []
for s in eval_dataset:
    user_msg = s['messages'][1]['content']
    if 'Team A' in user_msg and 'Team B' in user_msg:
        anon_eval.append(s)
    else:
        named_eval.append(s)
print(f"  named: {len(named_eval)}")
print(f"  anon:  {len(anon_eval)}")

Eval samples:  128
Train samples: 384
  named: 64
  anon:  64


## 3. Prediction parser

Reused verbatim from `eval_harness.ipynb` — score overrides text label when score is parsed, which fixes the known failure mode where the model outputs "home_win" with a "0-1" score.


In [4]:
def parse_prediction(text):
    """Extract match result and score from model output text.

    Score is considered more precise than text label — if a score is parsed,
    it overrides whatever result label the model emitted.
    """
    result = None
    home_goals = None
    away_goals = None

    text_lower = text.lower()

    # Parse result from 'Prediction:' line
    pred_match = re.search(r'prediction[:\s]+(.*?)(?:\n|$)', text_lower)
    if pred_match:
        pred_text = pred_match.group(1).strip()
        if 'draw' in pred_text:
            result = 'draw'
        elif 'home' in pred_text or 'team a' in pred_text:
            result = 'home_win'
        elif 'away' in pred_text or 'team b' in pred_text:
            result = 'away_win'
        elif 'win' in pred_text:
            result = 'has_winner'

    # Parse score — try "Score: N-M" first, then bare "N-M"
    score_patterns = [
        r'score[:\s]+.*?(\d+)\s*[-\u2013]\s*(\d+)',
        r'(\d+)\s*[-\u2013]\s*(\d+)',
    ]
    for pattern in score_patterns:
        m = re.search(pattern, text)
        if m:
            home_goals = int(m.group(1))
            away_goals = int(m.group(2))
            break

    # Override result from score when available
    if home_goals is not None and away_goals is not None:
        if home_goals > away_goals:
            result = 'home_win'
        elif home_goals < away_goals:
            result = 'away_win'
        else:
            result = 'draw'

    return {
        'result': result,
        'home_goals': home_goals,
        'away_goals': away_goals,
        'parsed': result is not None,
    }


def parse_ground_truth(assistant_msg):
    return parse_prediction(assistant_msg)


def compute_metrics(results):
    total = len(results)
    parsed = sum(1 for r in results if r['pred']['parsed'])
    correct_result, correct_score = 0, 0
    h_err, a_err = [], []
    for r in results:
        if not r['pred']['parsed'] or not r['gt']['parsed']:
            continue
        if r['pred']['result'] == r['gt']['result']:
            correct_result += 1
        if (r['pred']['home_goals'] == r['gt']['home_goals']
                and r['pred']['away_goals'] == r['gt']['away_goals']):
            correct_score += 1
        if r['pred']['home_goals'] is not None and r['gt']['home_goals'] is not None:
            h_err.append(abs(r['pred']['home_goals'] - r['gt']['home_goals']))
            a_err.append(abs(r['pred']['away_goals'] - r['gt']['away_goals']))
    return {
        'total': total,
        'parsed': parsed,
        'parse_rate': parsed / total if total else 0,
        'result_accuracy': correct_result / parsed if parsed else 0,
        'score_exact_match': correct_score / parsed if parsed else 0,
        'goal_mae': float(np.mean(h_err + a_err)) if h_err else float('nan'),
    }

print("Parser + metrics defined.")

Parser + metrics defined.


## 4. Select 5-shot demonstrations

Stratified 2 home / 2 away / 1 draw from the **named** training subset. Because train.jsonl contains both named and anonymized versions of the same matches, we draw demos only from the named half, then swap them for the Team A/B version when the eval sample is anonymized. This keeps demo *content* identical across the two conditions — only the team-name rendering differs.


In [5]:
import random
random.seed(42)

# Split train into named / anon
train_named, train_anon = [], []
for s in train_dataset:
    if 'Team A' in s['messages'][1]['content'] and 'Team B' in s['messages'][1]['content']:
        train_anon.append(s)
    else:
        train_named.append(s)
print(f"Train named: {len(train_named)}, anon: {len(train_anon)}")

# Bucket named train by ground-truth label
def get_gt_label(sample):
    return parse_ground_truth(sample['messages'][2]['content'])['result']

buckets = {'home_win': [], 'away_win': [], 'draw': []}
for i, s in enumerate(train_named):
    lbl = get_gt_label(s)
    if lbl in buckets:
        buckets[lbl].append(i)
print("Named train by label:", {k: len(v) for k, v in buckets.items()})

# Stratified sample: 2 home / 2 away / 1 draw
demo_indices_named = (
    random.sample(buckets['home_win'], 2)
    + random.sample(buckets['away_win'], 2)
    + random.sample(buckets['draw'], 1)
)
random.shuffle(demo_indices_named)

# Find the parallel anonymized version of each named demo.
# Matches by prior-season aggregated stats being identical and position in the dataset.
# Simplest reliable approach: train.jsonl is an alternating pattern [named_i, anon_i] OR [named..., anon...].
# We verify empirically here.
demos_named = [train_named[i] for i in demo_indices_named]

# Try to find the anon counterpart by matching the assistant message (ground truth identifies the match)
def find_anon_pair(named_sample, anon_pool):
    gt_n = named_sample['messages'][2]['content']
    # Normalize both GTs — remove team names — by finding numeric score + result pattern
    pred_n = parse_ground_truth(gt_n)
    # Look for anon sample with same result + score + roughly similar user prompt length
    # (same match produces identical anonymized stats, so stats in prompt will match)
    # Quick heuristic: match on score + result
    candidates = [
        a for a in anon_pool
        if parse_ground_truth(a['messages'][2]['content'])['result'] == pred_n['result']
        and parse_ground_truth(a['messages'][2]['content'])['home_goals'] == pred_n['home_goals']
        and parse_ground_truth(a['messages'][2]['content'])['away_goals'] == pred_n['away_goals']
    ]
    return candidates[0] if candidates else None

demos_anon = []
for d in demos_named:
    pair = find_anon_pair(d, train_anon)
    if pair is None:
        # Fallback: just draw a random anon with the same label
        lbl = parse_ground_truth(d['messages'][2]['content'])['result']
        matches = [a for a in train_anon if parse_ground_truth(a['messages'][2]['content'])['result'] == lbl]
        pair = random.choice(matches) if matches else d  # last-resort fallback
    demos_anon.append(pair)

print(f"\nSelected {len(demos_named)} demos.")
for i, d in enumerate(demos_named):
    lbl = parse_ground_truth(d['messages'][2]['content'])['result']
    print(f"  Demo {i}: {lbl}")

Train named: 192, anon: 192
Named train by label: {'home_win': 77, 'away_win': 73, 'draw': 42}

Selected 5 demos.
  Demo 0: draw
  Demo 1: away_win
  Demo 2: away_win
  Demo 3: home_win
  Demo 4: home_win


## 5. Load base model (4-bit, same config as eval_harness)

In [6]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model.eval()
print(f"Base model loaded. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Base model loaded. GPU: 5.70 GB


## 6. ICL (5-shot) evaluation

The eval sample's own system message is preserved. We prepend 5 demonstration (user, assistant) turns before the eval user turn. Demos are swapped to anonymized versions when the eval sample is anonymized.


In [7]:
def build_icl_messages(demos, eval_sample):
    '''Build ICL prompt: eval sample's system message + demo turns + eval user turn.'''
    system_msg = eval_sample['messages'][0]
    messages = [system_msg]
    for d in demos:
        messages.append({"role": "user", "content": d['messages'][1]['content']})
        messages.append({"role": "assistant", "content": d['messages'][2]['content']})
    messages.append({"role": "user", "content": eval_sample['messages'][1]['content']})
    return messages


def run_icl(samples, demos_n, demos_a, model, tokenizer, model_name="ICL"):
    '''demos_n = named demos, demos_a = anonymized demos; chosen per sample.'''
    results = []
    model.eval()
    for i, sample in enumerate(samples):
        is_anon = ('Team A' in sample['messages'][1]['content']
                   and 'Team B' in sample['messages'][1]['content'])
        demos = demos_a if is_anon else demos_n
        messages = build_icl_messages(demos, sample)
        gt = parse_ground_truth(sample['messages'][2]['content'])

        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            input_text, return_tensors="pt", truncation=True, max_length=2048
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.1,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1,
            )
        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )
        pred = parse_prediction(response)
        results.append({
            'sample_idx': i,
            'is_anon': is_anon,
            'gt': gt,
            'pred': pred,
            'raw_output': response[:500],
        })
        if (i + 1) % 10 == 0:
            print(f"  [{model_name}] {i+1}/{len(samples)} done")
    return results

print("Running ICL on full eval set (128 samples)...")
icl_results = run_icl(eval_dataset, demos_named, demos_anon, model, tokenizer, "ICL")

# Save raw results
with open("icl_results.json", "w") as f:
    json.dump(icl_results, f, indent=2, default=str)
print(f"Saved icl_results.json ({len(icl_results)} preds)")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running ICL on full eval set (128 samples)...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  [ICL] 10/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

  [ICL] 110/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  [ICL] 120/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Saved icl_results.json (128 preds)


## 7. Chain-of-Thought (5-step) evaluation

No few-shot demonstrations. We append a 5-step reasoning scaffold to the eval sample's system message, then run the base model. The scaffold explicitly asks the model to compare goal output → top scorer → defense → weigh conflicts → predict.


In [8]:
COT_SCAFFOLD = (
    "\n\nBefore your final answer, reason step-by-step:\n"
    "Step 1: Compare the two teams' squad goal output. Which is higher and by how much?\n"
    "Step 2: Compare the quality of each team's top scorer.\n"
    "Step 3: Compare defensive stats (tackles, clean sheets, defensive rating).\n"
    "Step 4: Identify any conflicts between these signals and weigh them.\n"
    "Step 5: Based on the above, predict the outcome.\n\n"
    "After your reasoning, conclude in the same format as requested above."
)


def build_cot_messages(eval_sample):
    system_msg = {
        "role": "system",
        "content": eval_sample['messages'][0]['content'] + COT_SCAFFOLD,
    }
    return [system_msg, eval_sample['messages'][1]]


def run_cot(samples, model, tokenizer, model_name="CoT"):
    results = []
    model.eval()
    for i, sample in enumerate(samples):
        is_anon = ('Team A' in sample['messages'][1]['content']
                   and 'Team B' in sample['messages'][1]['content'])
        messages = build_cot_messages(sample)
        gt = parse_ground_truth(sample['messages'][2]['content'])

        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            input_text, return_tensors="pt", truncation=True, max_length=1024
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,  # more tokens for CoT reasoning
                temperature=0.1,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1,
            )
        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )
        pred = parse_prediction(response)
        results.append({
            'sample_idx': i,
            'is_anon': is_anon,
            'gt': gt,
            'pred': pred,
            'raw_output': response[:800],
        })
        if (i + 1) % 10 == 0:
            print(f"  [{model_name}] {i+1}/{len(samples)} done")
    return results

print("Running CoT on full eval set (128 samples)...")
cot_results = run_cot(eval_dataset, model, tokenizer, "CoT")

with open("cot_results.json", "w") as f:
    json.dump(cot_results, f, indent=2, default=str)
print(f"Saved cot_results.json ({len(cot_results)} preds)")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running CoT on full eval set (128 samples)...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  [CoT] 10/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

  [CoT] 110/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  [CoT] 120/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Saved cot_results.json (128 preds)


## 8. Metrics + named/anon breakdown

Direct comparison with the existing FT (52.3% named 50.0% / anon 54.7%) and base (45.3% overall) numbers.


In [9]:
def split_metrics(results):
    named = [r for r in results if not r['is_anon']]
    anon  = [r for r in results if r['is_anon']]
    return {
        'overall': compute_metrics(results),
        'named':   compute_metrics(named),
        'anon':    compute_metrics(anon),
    }

icl_metrics = split_metrics(icl_results)
cot_metrics = split_metrics(cot_results)

def pct(x): return f"{x*100:.1f}%" if isinstance(x, float) and not np.isnan(x) else str(x)

print("=" * 80)
print(f"{'Condition':<30} {'Parse%':>8} {'Result Acc':>12} {'Score EM':>10} {'Goal MAE':>10}")
print("=" * 80)
rows = [
    ("ICL (overall, 5-shot)",   icl_metrics['overall']),
    ("  ICL named",              icl_metrics['named']),
    ("  ICL anonymized",         icl_metrics['anon']),
    ("CoT (overall, 5-step)",   cot_metrics['overall']),
    ("  CoT named",              cot_metrics['named']),
    ("  CoT anonymized",         cot_metrics['anon']),
]
for name, m in rows:
    print(f"{name:<30} {pct(m['parse_rate']):>8} {pct(m['result_accuracy']):>12} "
          f"{pct(m['score_exact_match']):>10} "
          f"{m['goal_mae']:.2f}" if not np.isnan(m['goal_mae']) else "    N/A")
print("=" * 80)

# Save combined summary
summary = {'icl': icl_metrics, 'cot': cot_metrics}
with open("baseline_metrics.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print("\nSaved baseline_metrics.json")

Condition                        Parse%   Result Acc   Score EM   Goal MAE
ICL (overall, 5-shot)            100.0%        49.2%      10.9% 0.98
  ICL named                      100.0%        50.0%      14.1% 0.93
  ICL anonymized                 100.0%        48.4%       7.8% 1.04
CoT (overall, 5-step)            100.0%        52.3%      11.7% 3.19
  CoT named                      100.0%        51.6%      12.5% 5.40
  CoT anonymized                 100.0%        53.1%      10.9% 0.98

Saved baseline_metrics.json


## 9. Cross-reference with existing results

Plug these numbers into the table for the final report:

| Condition        | Named | Anonymized | Diff (anon − named) |
|------------------|:-----:|:----------:|:-------------------:|
| Base Llama 3.1 8B | 45.3% (hardcoded from prior run; full eval not split) | — | — |
| **ICL (5-shot)** | __ | __ | __ |
| **CoT (5-step)** | __ | __ | __ |
| **FT (QLoRA, n=384)** | 50.0% | 54.7% | **+4.7** |
| Always home win  | 45.3% | 45.3% | 0.0 |

**Key pattern to look for:**
- **FT shows anon > named** (existing finding). If ICL and CoT **don't** show this pattern, that's direct evidence that de-biasing requires parameter updates — the central thesis of the paper.
- **FT ≫ ICL overall but FT ≈ ICL on anonymized** would be the strongest possible evidence: it means FT's only real contribution over ICL is in the named setting, which is exactly the setting where pre-training priors need to be overridden.
